# Stage B - B2 Baseline Prediction Cache

Notebook version of `scripts/run_stage_b_b2_baseline_cache.py`.

Use this notebook after Stage A has produced and locked `frozen_validation_manifest.v1` with a passing gate report. The notebook keeps environment-specific choices in one CONFIG cell and writes the required Stage B artifacts under `artifacts/baseline_predictions_cache/<RUN_ID>/`.


## Plan Contract Check

This notebook is aligned with `document/plan/plan_first.md` and `document/plan/Data_Plan.md`:

- Uses `run_id=b2_stage2_gold__crop-none__prompt-v1__val-v1`, `checkpoint_id=b2_stage2_gold`, `prompt_version=prompt-v1`, `artifact_version=frozen_validation_manifest.v1`, `crop_mode=none`, and `metric_version=official-compatible-v1`.
- Loads only the locked Stage A manifest and writes `source_manifest=frozen_validation_manifest.v1` plus the manifest path/checksum into config and score files.
- Saves raw model output before parsing, then writes parsed `validation_predictions.csv` with the required columns.
- Writes `config.json`, `validation_predictions.csv`, `validation_raw_outputs.jsonl`, `validation_score.json`, `failed_rows.jsonl`, and `runtime_summary.json`.
- Uses fixed baseline generation params: `do_sample=False`, `num_beams=1`, `max_new_tokens_page=4096`, `max_new_tokens_crop=0`, `max_pixels_crop=not_used_for_crop_none`.
- Computes the parse-fail gate threshold as `max(3 rows, ceil(0.5% * manifest rows))` and records pass-gate details in `validation_score.json`.
- Appends the B2 baseline row to `artifacts/ablations/ablation_results.csv` with `decision=keep` when scoring completes.


## Imports

Load standard libraries and pandas. The notebook assumes common Kaggle/ML dependencies are already available; it does not install packages or search many candidate paths.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import subprocess
import sys
import time
from collections import OrderedDict
from contextlib import nullcontext
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display
from tqdm.auto import tqdm


## Clone Project Repo

Kaggle terminal-style setup. This always prepares the lightweight artifact repo under `/kaggle/working/htr-artifacts`; rerunning the cell pulls the latest changes.


In [ ]:
%%bash
set -euo pipefail

REPO_URL="https://github.com/Quii29/htr-artifacts.git"
REPO_BRANCH=""  # optional, e.g. main
PROJECT_ROOT="/kaggle/working/htr-artifacts"

if [ -d "$PROJECT_ROOT/.git" ]; then
  git -C "$PROJECT_ROOT" fetch --all --prune
  if [ -n "$REPO_BRANCH" ]; then
    git -C "$PROJECT_ROOT" checkout "$REPO_BRANCH"
    git -C "$PROJECT_ROOT" pull --ff-only origin "$REPO_BRANCH"
  else
    git -C "$PROJECT_ROOT" pull --ff-only
  fi
else
  if [ -n "$REPO_BRANCH" ]; then
    git clone --branch "$REPO_BRANCH" --single-branch "$REPO_URL" "$PROJECT_ROOT"
  else
    git clone "$REPO_URL" "$PROJECT_ROOT"
  fi
fi

echo "Project repo ready: $PROJECT_ROOT"


## Install Runtime Dependencies

Run this once after cloning the repo. It installs the packages needed for Qwen3-VL, PEFT LoRA loading, 4-bit quantization, and Qwen vision preprocessing. The `bitsandbytes>=0.46.1` line fixes the common Kaggle error when `LOAD_IN_4BIT=True`.


In [ ]:
%%bash
set -euo pipefail
python -m pip install -q -U   "bitsandbytes>=0.46.1"   accelerate   peft   qwen-vl-utils

echo "Runtime dependencies ready."


## Runtime Config

Only edit the values that affect the run: data mount, model mount, LoRA checkpoint, prompt source, and generation/runtime controls. The fixed Stage B contract values stay here too so `config.json` remains reproducible, but they normally should not change.


In [ ]:
# =============================================================================
# RUNTIME CONFIG - important values for Kaggle execution.
# =============================================================================

# Repo location prepared by the terminal clone cell above.
PROJECT_ROOT = Path("/kaggle/working/htr-artifacts").resolve()
WORKSPACE_ROOT = PROJECT_ROOT.parent
RUN_SOURCE = "kaggle_notebook"

# Fixed Stage B baseline identity from plan_first.md/Data_Plan.md.
RUN_ID = "b2_stage2_gold__crop-none__prompt-v1__val-v1"
CHECKPOINT_ID = "b2_stage2_gold"
PROMPT_VERSION = "prompt-v1"
ARTIFACT_VERSION = "frozen_validation_manifest.v1"
CROP_MODE = "none"
METRIC_VERSION = "official-compatible-v1"

# Locked Stage A manifest from the cloned repo.
MANIFEST_PATH = PROJECT_ROOT / "artifacts/manifests/frozen_validation_manifest.v1.jsonl"

# RUKOPYS data mount. This folder must contain train/images/.
DATASET_ROOT = Path("/kaggle/input/datasets/quii29/rukopys-dataset")
IMAGE_SPLIT = "train"

# Outputs stay in the cloned repo under /kaggle/working so they are writable.
OUTPUT_ROOT = PROJECT_ROOT / "artifacts/baseline_predictions_cache"
ABLATION_RESULTS_PATH = PROJECT_ROOT / "artifacts/ablations/ablation_results.csv"
WRITE_ABLATION_ROW = True

# Current base model: Qwen3-VL 8B Instruct, loaded from Hugging Face.
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

# B2 Stage2 Gold LoRA checkpoint, added as a Kaggle Dataset.
# This folder must contain adapter_config.json.
LORA_DIR: Path | None = Path("/kaggle/input/datasets/lhongthyan/htd-fine-tune-v2-dataset/qwen3vl_rukopys_stage2_gold/qwen3vl_rukopys_lora_final")

# Optional prompt JSON. If None, use LORA_DIR/rukopys_prompt_config.json when present,
# otherwise fallback to PAGE_PROMPT in the next section.
PROMPT_CONFIG_PATH: Path | None = None

# Main tuning knobs for this baseline run.
DEVICE = "cuda:0"
LOAD_IN_4BIT = True  # Requires bitsandbytes; use the dependency install cell above.
MAX_PIXELS_PAGE = 650_000
MAX_NEW_TOKENS_PAGE = 4096
SAVE_EVERY = 10
SCORE_ONLY = False

# Notebook progress logging. LOG_EVERY=1 logs every image; increase if too noisy.
LOG_EVERY = 1
LOG_IMAGE_START = True
LOG_IMAGE_DONE = True
GENERATION_HEARTBEAT_SEC = 30  # Prints "still generating" while one image is blocking.

# Decoding is fixed for the baseline contract. Sampling knobs are intentionally unset.
DO_SAMPLE = False
NUM_BEAMS = 1

# Safety/debug controls. Keep default for the official run.
DEBUG_LIMIT: int | None = None
ALLOW_CONFIG_MISMATCH = False


## Constants

Schema, prompt fallback, and allowed region types shared by the rest of the notebook.


In [ ]:
# =============================================================================
# Constants.
# =============================================================================

VALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}
SCORABLE_TYPES = {"handwritten", "printed", "formula", "table", "annotation"}
NON_TEXT_TYPES = {"image", "graph"}

PAGE_PROMPT = (
    "Extract every visible document region. Return only a compact JSON array. "
    "Each item must have keys bbox,type,text. bbox is [x1,y1,x2,y2] on a 0-1000 grid. "
    "type is one of handwritten,printed,formula,table,annotation,image,graph. "
    "Use empty text for image and graph. Preserve reading order."
)

PREDICTION_COLUMNS = [
    "image_id",
    "file_name",
    "submission_image",
    "regions",
    "parse_ok",
    "error_type",
    "raw_output_id",
    "checkpoint_id",
    "prompt_version",
    "runtime_sec",
]

RAW_OUTPUT_FIELDS = [
    "raw_output_id",
    "image_id",
    "file_name",
    "prompt_version",
    "generation_params",
    "raw_text",
    "created_at",
]


@dataclass(frozen=True)
class PromptSettings:
    page_prompt: str
    max_pixels_page: int
    prompt_config_path: str | None


@dataclass(frozen=True)
class ParseResult:
    regions: list[dict[str, Any]]
    parse_ok: bool
    error_type: str
    warnings: list[str]


## Small utilities

Small helpers for timestamps, stable JSON, checksums, atomic writes, and relative display paths.


In [ ]:
# =============================================================================
# Small utilities.
# =============================================================================


def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def display_path(path: Path) -> str:
    try:
        resolved = path.expanduser().resolve()
        return f"./{resolved.relative_to(WORKSPACE_ROOT.resolve()).as_posix()}"
    except ValueError:
        return str(path.expanduser().resolve())
    except OSError:
        return str(path)


def stable_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def raw_output_id(run_id: str, image_id: str) -> str:
    return hashlib.sha1(f"{run_id}|{image_id}".encode("utf-8")).hexdigest()[:20]


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)


def write_json(path: Path, value: Any) -> None:
    atomic_write_text(path, json.dumps(value, ensure_ascii=False, indent=2) + "\n")


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    text = "".join(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n" for row in rows)
    atomic_write_text(path, text)


def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n")
        f.flush()
        os.fsync(f.fileno())


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"{path}:{line_no}: invalid JSONL: {exc}") from exc
            if not isinstance(obj, dict):
                raise ValueError(f"{path}:{line_no}: expected JSON object")
            rows.append(obj)
    return rows


def git_commit() -> str | None:
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=PROJECT_ROOT,
            check=True,
            capture_output=True,
            text=True,
        )
    except Exception:
        return None
    return result.stdout.strip() or None


def normalize_posix(value: Any) -> str:
    text = str(value or "").strip().replace("\\", "/")
    while text.startswith("./"):
        text = text[2:]
    return text.strip("/")


def format_duration(seconds: float | int | None) -> str:
    seconds = float(seconds or 0.0)
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, sec = divmod(seconds, 60)
    if minutes < 60:
        return f"{int(minutes)}m {sec:04.1f}s"
    hours, minutes = divmod(minutes, 60)
    return f"{int(hours)}h {int(minutes):02d}m {sec:04.1f}s"


def run_with_heartbeat(label: str, interval_sec: int | float, fn: Any) -> Any:
    if not interval_sec or interval_sec <= 0:
        return fn()

    from threading import Event, Thread

    stop = Event()
    started = time.time()

    def heartbeat() -> None:
        while not stop.wait(float(interval_sec)):
            tqdm.write(f"{label} still running | elapsed={format_duration(time.time() - started)}")

    thread = Thread(target=heartbeat, daemon=True)
    thread.start()
    try:
        return fn()
    finally:
        stop.set()
        thread.join(timeout=1)


def show_phase(title: str, detail: str | None = None) -> None:
    body = f"### {title}"
    if detail:
        body += f"\n\n{detail}"
    display(Markdown(body))


def show_table(title: str, rows: list[dict[str, Any]]) -> None:
    display(Markdown(f"### {title}"))
    display(pd.DataFrame(rows))


def show_run_overview(
    config: dict[str, Any],
    prompt: PromptSettings,
    paths: dict[str, Path],
    manifest_rows: list[dict[str, Any]],
) -> None:
    show_table(
        "Run Overview",
        [
            {"field": "run_id", "value": RUN_ID},
            {"field": "checkpoint_id", "value": CHECKPOINT_ID},
            {"field": "base_model", "value": MODEL_ID},
            {"field": "lora_dir", "value": display_path(LORA_DIR) if LORA_DIR is not None else "None"},
            {"field": "manifest", "value": display_path(MANIFEST_PATH)},
            {"field": "manifest_rows", "value": len(manifest_rows)},
            {"field": "dataset_root", "value": display_path(DATASET_ROOT)},
            {"field": "output_dir", "value": display_path(paths["run_dir"])},
            {"field": "prompt_source", "value": prompt.prompt_config_path or "CONFIG.PAGE_PROMPT"},
            {"field": "max_pixels_page", "value": prompt.max_pixels_page},
            {"field": "score_only", "value": SCORE_ONLY},
        ],
    )
    show_table(
        "Generation Params",
        [{"param": key, "value": value} for key, value in config["generation_params"].items()],
    )


def show_artifact_outputs(paths: dict[str, Path]) -> None:
    rows = []
    for name, path in paths.items():
        if name == "run_dir":
            continue
        rows.append(
            {
                "artifact": name,
                "path": display_path(path),
                "exists": path.exists(),
                "size_mb": round(path.stat().st_size / (1024 * 1024), 3) if path.exists() else None,
            }
        )
    show_table("Artifacts", rows)


def show_score_summary(score: dict[str, Any], runtime_summary: dict[str, Any]) -> None:
    gate = score.get("pass_gate", {})
    show_table(
        "Score Summary",
        [
            {"metric": "total_score", "value": score.get("total_score")},
            {"metric": "detection_f1", "value": score.get("detection_f1")},
            {"metric": "class_acc", "value": score.get("class_acc")},
            {"metric": "region_cer", "value": score.get("region_cer")},
            {"metric": "page_cer", "value": score.get("page_cer")},
            {"metric": "parse_fail_count", "value": score.get("parse_fail_count")},
            {"metric": "parse_fail_threshold", "value": score.get("parse_fail_threshold")},
            {"metric": "runtime_total", "value": format_duration(runtime_summary.get("runtime_total_sec"))},
            {"metric": "runtime_per_page", "value": f"{runtime_summary.get('runtime_per_page_sec', 0.0):.3f}s"},
        ],
    )
    show_table("Pass Gate", [{"check": key, "pass": value} for key, value in gate.items()])


## Runtime Config

Only edit the values that affect the run: data mount, model mount, LoRA checkpoint, prompt source, and generation/runtime controls. The fixed Stage B contract values stay here too so `config.json` remains reproducible, but they normally should not change.


In [ ]:
# =============================================================================
# Config and input validation.
# =============================================================================


def read_manifest(path: Path) -> list[dict[str, Any]]:
    rows = read_jsonl(path)
    required = {"image_id", "file_name", "submission_image", "source", "image_width", "image_height", "regions"}
    for idx, row in enumerate(rows):
        missing = required - set(row)
        if missing:
            raise ValueError(f"{path}: row {idx} missing required manifest fields: {sorted(missing)}")

    if DEBUG_LIMIT is not None:
        if RUN_ID == "b2_stage2_gold__crop-none__prompt-v1__val-v1":
            raise RuntimeError("DEBUG_LIMIT is set for the production RUN_ID. Use a debug RUN_ID first.")
        rows = rows[:DEBUG_LIMIT]
    return rows


def require_runtime_paths() -> None:
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(f"Manifest not found: {display_path(MANIFEST_PATH)}")
    if not SCORE_ONLY and not DATASET_ROOT.exists():
        raise FileNotFoundError(f"DATASET_ROOT not found: {display_path(DATASET_ROOT)}")
    if SCORE_ONLY:
        return
    if LORA_DIR is None:
        raise ValueError("SCORE_ONLY=False requires LORA_DIR in the CONFIG block.")
    if not (LORA_DIR / "adapter_config.json").exists():
        raise FileNotFoundError(f"adapter_config.json not found under LORA_DIR={display_path(LORA_DIR)}")
    if MODEL_ID.startswith("/") and not Path(MODEL_ID).exists():
        raise FileNotFoundError(f"MODEL_ID path not found: {MODEL_ID}")


def load_prompt_settings() -> PromptSettings:
    prompt_config_path = PROMPT_CONFIG_PATH
    if prompt_config_path is None and LORA_DIR is not None:
        lora_prompt_config = LORA_DIR / "rukopys_prompt_config.json"
        if lora_prompt_config.exists():
            prompt_config_path = lora_prompt_config

    if prompt_config_path is None:
        return PromptSettings(PAGE_PROMPT, MAX_PIXELS_PAGE, None)

    if not prompt_config_path.exists():
        raise FileNotFoundError(f"PROMPT_CONFIG_PATH not found: {display_path(prompt_config_path)}")
    cfg = json.loads(prompt_config_path.read_text(encoding="utf-8"))
    return PromptSettings(
        page_prompt=str(cfg.get("page_prompt") or PAGE_PROMPT),
        max_pixels_page=int(cfg.get("max_pixels_page") or MAX_PIXELS_PAGE),
        prompt_config_path=display_path(prompt_config_path),
    )


def output_paths() -> dict[str, Path]:
    run_dir = OUTPUT_ROOT / RUN_ID
    return {
        "run_dir": run_dir,
        "config": run_dir / "config.json",
        "predictions": run_dir / "validation_predictions.csv",
        "raw_outputs": run_dir / "validation_raw_outputs.jsonl",
        "score": run_dir / "validation_score.json",
        "failed_rows": run_dir / "failed_rows.jsonl",
        "runtime_summary": run_dir / "runtime_summary.json",
    }


def generation_params(prompt: PromptSettings) -> dict[str, Any]:
    return {
        "do_sample": DO_SAMPLE,
        "num_beams": NUM_BEAMS,
        "crop_mode": CROP_MODE,
        "max_new_tokens_page": MAX_NEW_TOKENS_PAGE,
        "max_new_tokens_crop": 0,
        "max_pixels_page": prompt.max_pixels_page,
        "max_pixels_crop": "not_used_for_crop_none",
    }


def runtime_controls() -> dict[str, Any]:
    return {
        "device": DEVICE,
        "load_in_4bit": LOAD_IN_4BIT,
        "score_only": SCORE_ONLY,
        "save_every": SAVE_EVERY,
        "log_every": LOG_EVERY,
        "generation_heartbeat_sec": GENERATION_HEARTBEAT_SEC,
    }


def validate_stage_b_contract(config: dict[str, Any]) -> None:
    expected_identity = {
        "run_id": "b2_stage2_gold__crop-none__prompt-v1__val-v1",
        "checkpoint_id": "b2_stage2_gold",
        "prompt_version": "prompt-v1",
        "artifact_version": "frozen_validation_manifest.v1",
        "crop_mode": "none",
        "metric_version": "official-compatible-v1",
        "source_manifest": "frozen_validation_manifest.v1",
    }
    generation = config.get("generation_params", {})
    expected_generation = {
        "do_sample": False,
        "num_beams": 1,
        "crop_mode": "none",
        "max_new_tokens_page": 4096,
        "max_new_tokens_crop": 0,
        "max_pixels_crop": "not_used_for_crop_none",
    }

    mismatches = []
    for key, expected in expected_identity.items():
        if config.get(key) != expected:
            mismatches.append(f"{key}={config.get(key)!r}, expected {expected!r}")
    for key, expected in expected_generation.items():
        if generation.get(key) != expected:
            mismatches.append(f"generation_params.{key}={generation.get(key)!r}, expected {expected!r}")
    if not isinstance(generation.get("max_pixels_page"), int) or generation["max_pixels_page"] <= 0:
        mismatches.append("generation_params.max_pixels_page must be a positive integer")

    if mismatches:
        raise RuntimeError("Stage B baseline contract mismatch:\n- " + "\n- ".join(mismatches))


def config_fingerprint(config: dict[str, Any]) -> str:
    critical = {
        "run_id": config["run_id"],
        "checkpoint_id": config["checkpoint_id"],
        "prompt_version": config["prompt_version"],
        "artifact_version": config["artifact_version"],
        "crop_mode": config["crop_mode"],
        "source_manifest_sha256": config["source_manifest_sha256"],
        "prompt_sha256": config["prompt_sha256"],
        "generation_params": config["generation_params"],
        "model_id": config["model_id"],
        "lora_dir": config["lora_dir"],
    }
    return sha256_text(stable_json(critical))


def build_config(manifest_rows: list[dict[str, Any]], prompt: PromptSettings, paths: dict[str, Path]) -> dict[str, Any]:
    config: dict[str, Any] = {
        "run_id": RUN_ID,
        "stage": "Stage B - B2 Baseline Prediction Cache",
        "status": "running",
        "created_at": utc_now(),
        "run_source": RUN_SOURCE,
        "script_path": RUN_SOURCE,
        "script_sha256": None,
        "code_version": git_commit(),
        "checkpoint_id": CHECKPOINT_ID,
        "prompt_version": PROMPT_VERSION,
        "artifact_version": ARTIFACT_VERSION,
        "crop_mode": CROP_MODE,
        "metric_version": METRIC_VERSION,
        "source_manifest": ARTIFACT_VERSION,
        "source_manifest_path": display_path(MANIFEST_PATH),
        "source_manifest_sha256": sha256_file(MANIFEST_PATH),
        "manifest_row_count": len(manifest_rows),
        "dataset_root": display_path(DATASET_ROOT),
        "image_split": IMAGE_SPLIT,
        "model_id": MODEL_ID,
        "lora_dir": display_path(LORA_DIR) if LORA_DIR is not None else None,
        "prompt_config_path": prompt.prompt_config_path,
        "prompt_text": prompt.page_prompt,
        "prompt_sha256": sha256_text(prompt.page_prompt),
        "generation_params": generation_params(prompt),
        "runtime_controls": runtime_controls(),
        "prediction_schema": PREDICTION_COLUMNS,
        "raw_output_schema": RAW_OUTPUT_FIELDS,
        "outputs": {name: display_path(path) for name, path in paths.items() if name != "run_dir"},
        "resume_policy": "reuse valid cached predictions; parse existing raw outputs; generate only missing raw outputs",
        "debug_limit": DEBUG_LIMIT,
        "save_every": SAVE_EVERY,
    }
    config["config_fingerprint"] = config_fingerprint(config)
    return config


def write_or_validate_config(path: Path, config: dict[str, Any]) -> dict[str, Any]:
    if not path.exists():
        write_json(path, config)
        return config

    old = json.loads(path.read_text(encoding="utf-8"))
    old_fp = old.get("config_fingerprint")
    new_fp = config["config_fingerprint"]
    if old_fp and old_fp != new_fp and not ALLOW_CONFIG_MISMATCH:
        raise RuntimeError(
            "Existing config.json does not match the current CONFIG block. "
            "Use a new RUN_ID or set ALLOW_CONFIG_MISMATCH=True intentionally."
        )
    old.update({"status": "running", "resumed_at": utc_now(), "runtime_controls": runtime_controls()})
    write_json(path, old)
    return old


## Cache I/O

Reads/writes predictions, raw model outputs, and failed row diagnostics. Raw outputs are kept separately so parsing can be rerun without model inference.


In [ ]:
# =============================================================================
# Cache I/O.
# =============================================================================


def read_existing_predictions(path: Path) -> OrderedDict[str, dict[str, Any]]:
    rows: OrderedDict[str, dict[str, Any]] = OrderedDict()
    if not path.exists():
        return rows

    try:
        frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    except pd.errors.EmptyDataError:
        return rows
    missing = set(PREDICTION_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f"{display_path(path)} missing prediction columns: {sorted(missing)}")

    frame = frame.drop_duplicates(subset=["image_id"], keep="last")
    for row in frame[PREDICTION_COLUMNS].to_dict("records"):
        image_id = str(row.get("image_id") or "")
        if image_id:
            rows[image_id] = row
    return rows


def write_predictions_csv(path: Path, predictions: OrderedDict[str, dict[str, Any]], manifest_rows: list[dict[str, Any]]) -> None:
    ordered = [predictions[row["image_id"]] for row in manifest_rows if row["image_id"] in predictions]
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    pd.DataFrame(ordered, columns=PREDICTION_COLUMNS).to_csv(tmp, index=False)
    os.replace(tmp, path)


def read_raw_outputs(path: Path) -> OrderedDict[str, dict[str, Any]]:
    rows: OrderedDict[str, dict[str, Any]] = OrderedDict()
    if not path.exists():
        return rows
    for row in read_jsonl(path):
        rid = str(row.get("raw_output_id") or "")
        if rid:
            rows[rid] = row
    return rows


def write_failed_rows(path: Path, predictions: OrderedDict[str, dict[str, Any]], raw_rows: OrderedDict[str, dict[str, Any]]) -> None:
    failed: list[dict[str, Any]] = []
    for row in predictions.values():
        if str(row.get("parse_ok", "")).lower() == "true":
            continue
        raw = raw_rows.get(str(row.get("raw_output_id") or ""), {})
        failed.append(
            {
                "image_id": row.get("image_id"),
                "file_name": row.get("file_name"),
                "submission_image": row.get("submission_image"),
                "raw_output_id": row.get("raw_output_id"),
                "error_type": row.get("error_type"),
                "raw_text_preview": str(raw.get("raw_text") or "")[:1000],
            }
        )
    write_jsonl(path, failed)


## Parsing model output

Converts model text into cleaned region lists: JSON extraction, bbox normalization, type normalization, sorting, and de-duplication.


In [ ]:
# =============================================================================
# Parsing model output.
# =============================================================================


def strip_code_fences(text: str) -> str:
    text = (text or "").strip()
    if text.startswith("```json"):
        text = text[7:]
    elif text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    return text.strip()


def extract_json_array(raw_text: str) -> tuple[list[Any] | None, str]:
    clean = strip_code_fences(raw_text)
    candidates = [clean]
    match = re.search(r"\[.*\]", clean, flags=re.DOTALL)
    if match:
        candidates.append(match.group(0))

    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
        except Exception:
            continue
        if isinstance(parsed, list):
            return parsed, ""
        if isinstance(parsed, dict):
            for key in ("regions", "data", "items", "results"):
                if isinstance(parsed.get(key), list):
                    return parsed[key], ""

    objects: list[Any] = []
    for match in re.finditer(r"\{[^{}]*\"bbox\"[^{}]*\}", raw_text or "", flags=re.DOTALL):
        try:
            obj = json.loads(match.group(0))
        except Exception:
            continue
        if isinstance(obj, dict):
            objects.append(obj)
    if objects:
        return objects, ""
    return None, "json_array_not_found"


def numeric_bbox(box: Any) -> list[float] | None:
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    out: list[float] = []
    for item in box:
        if isinstance(item, bool):
            return None
        try:
            out.append(float(item))
        except Exception:
            return None
    return out


def bbox_to_pixels(box: Any, width: int, height: int) -> list[int] | None:
    values = numeric_bbox(box)
    if values is None:
        return None

    max_val = max(values)
    if max_val <= 1.5:
        x1, y1, x2, y2 = values[0] * width, values[1] * height, values[2] * width, values[3] * height
    elif max_val <= 1005:
        x1, y1, x2, y2 = values[0] / 1000 * width, values[1] / 1000 * height, values[2] / 1000 * width, values[3] / 1000 * height
    else:
        x1, y1, x2, y2 = values

    x1, x2 = sorted((max(0.0, min(float(width), x1)), max(0.0, min(float(width), x2))))
    y1, y2 = sorted((max(0.0, min(float(height), y1)), max(0.0, min(float(height), y2))))
    if x2 - x1 < 3 or y2 - y1 < 3:
        return None
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]


def normalize_region_type(value: Any) -> str:
    text = str(value or "handwritten").strip().lower()
    return text if text in VALID_TYPES else "handwritten"


def compute_iou(a: list[int] | list[float], b: list[int] | list[float]) -> float:
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    return inter / max(1, area_a + area_b - inter)


def parse_and_clean_regions(raw_text: str, record: dict[str, Any]) -> ParseResult:
    parsed, error = extract_json_array(raw_text)
    if parsed is None:
        return ParseResult(regions=[], parse_ok=False, error_type=error, warnings=[])

    width = int(record.get("image_width") or 1)
    height = int(record.get("image_height") or 1)
    cleaned: list[dict[str, Any]] = []
    warnings: list[str] = []

    for idx, item in enumerate(parsed):
        if not isinstance(item, dict):
            warnings.append(f"region_{idx}_not_object")
            continue
        bbox = bbox_to_pixels(item.get("bbox"), width, height)
        if bbox is None:
            warnings.append(f"region_{idx}_bad_bbox")
            continue
        rtype = normalize_region_type(item.get("type"))
        text = "" if rtype in NON_TEXT_TYPES else str(item.get("text") or "")
        cleaned.append({"bbox": bbox, "type": rtype, "text": text})

    cleaned.sort(key=lambda r: (r["bbox"][1], r["bbox"][0]))
    deduped: list[dict[str, Any]] = []
    for region in cleaned:
        duplicate = False
        for old in deduped:
            if compute_iou(region["bbox"], old["bbox"]) > 0.88:
                duplicate = True
                if len(str(region.get("text") or "")) > len(str(old.get("text") or "")):
                    old.update(region)
                break
        if not duplicate:
            deduped.append(region)

    return ParseResult(regions=deduped, parse_ok=True, error_type="", warnings=warnings)


## Standalone official-compatible metric

Self-contained scoring helpers: text normalization, IoU matching, CER, page CER, and composite score.


In [ ]:
# =============================================================================
# Standalone official-compatible metric.
# =============================================================================


LATEX_SYMBOLS = {
    r"\alpha": "α",
    r"\beta": "β",
    r"\gamma": "γ",
    r"\delta": "δ",
    r"\pi": "π",
    r"\theta": "θ",
    r"\lambda": "λ",
    r"\mu": "μ",
    r"\sqrt": "√",
    r"\cdot": "·",
    r"\times": "×",
    r"\div": "÷",
    r"\pm": "±",
    r"\leq": "≤",
    r"\le": "≤",
    r"\geq": "≥",
    r"\ge": "≥",
    r"\neq": "≠",
    r"\ne": "≠",
    r"\approx": "≈",
    r"\rightarrow": "→",
    r"\to": "→",
    r"\leftarrow": "←",
    r"\angle": "∠",
    r"\perp": "⊥",
    r"\parallel": "∥",
}

LOOKALIKE_LATIN_TO_CYRILLIC = str.maketrans(
    {
        "A": "А",
        "B": "В",
        "C": "С",
        "E": "Е",
        "H": "Н",
        "I": "І",
        "K": "К",
        "M": "М",
        "O": "О",
        "P": "Р",
        "T": "Т",
        "X": "Х",
        "a": "а",
        "c": "с",
        "e": "е",
        "i": "і",
        "o": "о",
        "p": "р",
        "x": "х",
        "y": "у",
    }
)

SUPERSCRIPTS = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")
SUBSCRIPTS = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")


def normalize_fraction_once(text: str) -> str:
    return re.sub(r"\\frac\{([^{}]+)\}\{([^{}]+)\}", r"\1/\2", text)


def normalize_text(text: Any, region_type: str = "handwritten") -> str:
    text = str(text or "")
    text = re.sub(r"~~([^~{}]+)~~\{([^{}]+)\}", r"\2", text)
    text = re.sub(r"~~([^~]+)~~", r"\1", text)
    text = re.sub(r"\\text\{([^}]*)\}", r"\1", text)
    text = re.sub(r"\\(left|right)\s*([()|\[\]{}.])", r"\2", text)
    text = re.sub(r"\\[,;:!]|\\quad|\\qquad|\\hspace\{[^}]*\}", " ", text)

    for _ in range(4):
        new = normalize_fraction_once(text)
        if new == text:
            break
        text = new

    for key in sorted(LATEX_SYMBOLS, key=len, reverse=True):
        text = text.replace(key, LATEX_SYMBOLS[key])

    text = re.sub(r"\\sqrt\{([^{}]+)\}", r"√\1", text)
    text = re.sub(r"\\(bar|hat|vec|overline|widetilde)\{([^{}]+)\}", r"\2", text)
    text = re.sub(r"\^\{([^{}])\}", r"^\1", text)
    text = re.sub(r"_\{([^{}])\}", r"_\1", text)
    text = text.translate(SUPERSCRIPTS).translate(SUBSCRIPTS)
    text = re.sub(r"[\u0300-\u036f]", "", text)

    text = text.translate(LOOKALIKE_LATIN_TO_CYRILLIC)
    text = re.sub(r"[\u2010-\u2015−]", "-", text)
    text = text.translate(str.maketrans({"«": '"', "»": '"', "“": '"', "”": '"', "„": '"', "ʼ": "'", "’": "'", "`": "'"}))
    text = text.replace("\u00a0", " ")

    if region_type == "formula":
        text = text.replace("*", "·").replace("⋅", "·").replace("∗", "·")
    if region_type == "table":
        text = re.sub(r"\s*\|\s*", "|", text)

    return re.sub(r"\s+", " ", text).strip()


def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i]
        for j, cb in enumerate(b, start=1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]


def greedy_match(gt_regions: list[dict[str, Any]], pred_regions: list[dict[str, Any]], threshold: float = 0.5) -> list[tuple[int, int]]:
    pairs: list[tuple[float, int, int]] = []
    for gi, gt in enumerate(gt_regions):
        for pi, pred in enumerate(pred_regions):
            if "bbox" in gt and "bbox" in pred:
                pairs.append((compute_iou(gt["bbox"], pred["bbox"]), gi, pi))
    pairs.sort(reverse=True)

    used_gt: set[int] = set()
    used_pred: set[int] = set()
    matched: list[tuple[int, int]] = []
    for score_iou, gi, pi in pairs:
        if score_iou >= threshold and gi not in used_gt and pi not in used_pred:
            used_gt.add(gi)
            used_pred.add(pi)
            matched.append((gi, pi))
    return matched


def is_scorable(region: dict[str, Any]) -> bool:
    return str(region.get("type") or "handwritten").lower() in SCORABLE_TYPES


def page_text(regions: list[dict[str, Any]]) -> str:
    scorable = [r for r in regions if is_scorable(r)]
    scorable.sort(key=lambda r: (r.get("bbox", [0, 0, 0, 0])[1], r.get("bbox", [0, 0, 0, 0])[0]))
    return "\n".join(normalize_text(r.get("text", ""), str(r.get("type") or "handwritten")) for r in scorable)


def score_detailed(solution_rows: list[dict[str, Any]], submission_rows: list[dict[str, Any]]) -> dict[str, Any]:
    sub_lookup = {row["image"]: json.loads(row["regions"]) for row in submission_rows}
    det_tp = det_fp = det_fn = class_correct = 0
    region_cers: list[float] = []
    page_cers: list[float] = []

    for row in solution_rows:
        gt = json.loads(row["regions"])
        pred = sub_lookup.get(row["image"], [])
        matched = greedy_match(gt, pred)
        det_tp += len(matched)
        det_fn += len(gt) - len(matched)
        det_fp += len(pred) - len(matched)

        for gi, pi in matched:
            gt_region = gt[gi]
            pred_region = pred[pi]
            gt_type = str(gt_region.get("type") or "handwritten").lower()
            pred_type = str(pred_region.get("type") or "handwritten").lower()
            class_correct += int(gt_type == pred_type)
            if is_scorable(gt_region):
                gt_text = normalize_text(gt_region.get("text", ""), gt_type)
                pred_text = normalize_text(pred_region.get("text", ""), pred_type)
                region_cers.append(levenshtein(pred_text, gt_text) / max(1, len(gt_text)))

        gt_page = page_text(gt)
        pred_page = page_text(pred)
        if gt_page:
            page_cers.append(levenshtein(pred_page, gt_page) / len(gt_page))

    precision = det_tp / max(1, det_tp + det_fp)
    recall = det_tp / max(1, det_tp + det_fn)
    detection_f1 = 2 * precision * recall / max(1e-12, precision + recall)
    class_acc = class_correct / max(1, det_tp)
    region_cer = sum(region_cers) / len(region_cers) if region_cers else 1.0
    page_cer = sum(page_cers) / len(page_cers) if page_cers else 1.0
    composite = (
        0.15 * detection_f1
        + 0.05 * class_acc
        + 0.30 * max(0.0, 1.0 - region_cer)
        + 0.50 * max(0.0, 1.0 - page_cer)
    )

    return {
        "composite_score": float(composite),
        "detection_f1": float(detection_f1),
        "class_acc": float(class_acc),
        "region_cer": float(region_cer),
        "page_cer": float(page_cer),
        "n_matched_regions": int(det_tp),
        "n_gt_regions": int(det_tp + det_fn),
        "n_pred_regions": int(det_tp + det_fp),
    }


## Model loading and generation

Loads Qwen3-VL + LoRA and runs page-level crop-none inference with the configured prompt. The baseline uses greedy decoding, so sampling-only fields from the pretrained `generation_config` are unset to avoid repeated Transformers warnings. A heartbeat message is printed while a single `model.generate()` call is still running.


In [ ]:
# =============================================================================
# Model loading and generation.
# =============================================================================


def configure_processor_for_generation(processor: Any) -> Any:
    tokenizer = processor.tokenizer
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    return processor


def configure_torch_runtime(torch: Any) -> None:
    if not DEVICE.startswith("cuda") or not torch.cuda.is_available():
        return
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")


def configure_generation_for_baseline(model: Any, processor: Any) -> None:
    generation_config = model.generation_config
    generation_config.do_sample = DO_SAMPLE
    generation_config.num_beams = NUM_BEAMS
    generation_config.max_new_tokens = MAX_NEW_TOKENS_PAGE
    generation_config.pad_token_id = processor.tokenizer.pad_token_id

    # Qwen generation_config may carry sampling knobs from pretraining. They are
    # invalid for this greedy baseline and cause repeated Transformers warnings.
    if not DO_SAMPLE:
        for name in (
            "temperature",
            "top_p",
            "top_k",
            "min_p",
            "top_h",
            "typical_p",
            "epsilon_cutoff",
            "eta_cutoff",
        ):
            if hasattr(generation_config, name):
                setattr(generation_config, name, None)


def load_model_and_processor(lora_dir: Path) -> tuple[Any, Any]:
    import torch
    from peft import PeftModel
    from transformers import AutoModelForImageTextToText, AutoProcessor

    configure_torch_runtime(torch)

    model_kwargs: dict[str, Any] = {
        "device_map": {"": DEVICE} if DEVICE.startswith("cuda") else "auto",
        "trust_remote_code": True,
        "attn_implementation": "sdpa",
        "low_cpu_mem_usage": True,
        "dtype": torch.float16 if DEVICE.startswith("cuda") else torch.float32,
    }
    if LOAD_IN_4BIT:
        from transformers import BitsAndBytesConfig

        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )

    total_start = time.time()
    tqdm.write(f"[model] loading base model: {MODEL_ID}")
    step_start = time.time()
    base = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **model_kwargs)
    tqdm.write(f"[model] base model loaded in {format_duration(time.time() - step_start)}")

    tqdm.write(f"[model] loading LoRA adapter: {display_path(lora_dir)}")
    step_start = time.time()
    model = PeftModel.from_pretrained(base, str(lora_dir), is_trainable=False)
    model.eval()
    tqdm.write(f"[model] LoRA adapter loaded in {format_duration(time.time() - step_start)}")

    tqdm.write(f"[model] loading processor/tokenizer from: {MODEL_ID}")
    step_start = time.time()
    processor = configure_processor_for_generation(AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True))
    configure_generation_for_baseline(model, processor)
    tqdm.write(f"[model] processor loaded in {format_duration(time.time() - step_start)}")

    tqdm.write(
        f"[model] generation config: max_new_tokens={model.generation_config.max_new_tokens}, "
        f"do_sample={model.generation_config.do_sample}, "
        f"num_beams={model.generation_config.num_beams}, "
        f"temperature={getattr(model.generation_config, 'temperature', None)}, "
        f"top_p={getattr(model.generation_config, 'top_p', None)}, "
        f"top_k={getattr(model.generation_config, 'top_k', None)}"
    )
    tqdm.write(f"[model] all model assets ready in {format_duration(time.time() - total_start)}")
    return model, processor


def page_messages(image_path: Path, prompt: PromptSettings) -> list[dict[str, Any]]:
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": str(image_path), "max_pixels": prompt.max_pixels_page},
                {"type": "text", "text": prompt.page_prompt},
            ],
        }
    ]


def generate_raw_text(
    model: Any,
    processor: Any,
    image_path: Path,
    prompt: PromptSettings,
    log_label: str | None = None,
) -> str:
    import torch
    from qwen_vl_utils import process_vision_info

    messages = page_messages(image_path, prompt)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    autocast_ctx = torch.amp.autocast("cuda", dtype=torch.float16) if DEVICE.startswith("cuda") else nullcontext()
    if DEVICE.startswith("cuda"):
        inputs = inputs.to(DEVICE)

    def run_generate() -> Any:
        return model.generate(
            **inputs,
            generation_config=model.generation_config,
        )

    with torch.inference_mode(), autocast_ctx:
        outputs = run_with_heartbeat(
            log_label or "model.generate",
            GENERATION_HEARTBEAT_SEC,
            run_generate,
        )

    trimmed = [output_ids[len(input_ids) :] for input_ids, output_ids in zip(inputs.input_ids, outputs)]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    del inputs, outputs, trimmed
    return decoded


## Pipeline outputs

Builds prediction rows, score files, runtime summary, ablation row, and the resumable inference loop. The loop now loads the model only when raw outputs are genuinely missing; otherwise it reuses valid cached predictions or parses existing raw outputs.


In [ ]:
# =============================================================================
# Pipeline outputs.
# =============================================================================


def resolve_image_path(record: dict[str, Any]) -> Path:
    file_name = normalize_posix(record["file_name"])
    direct = Path(file_name)
    if direct.is_absolute():
        return direct

    candidates = [
        DATASET_ROOT / IMAGE_SPLIT / file_name,
        DATASET_ROOT / file_name,
    ]
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]


def prediction_row(
    record: dict[str, Any],
    regions: list[dict[str, Any]],
    parse_ok: bool,
    error_type: str,
    rid: str,
    runtime_sec: float,
) -> dict[str, Any]:
    return {
        "image_id": record["image_id"],
        "file_name": record["file_name"],
        "submission_image": record["submission_image"],
        "regions": json.dumps(regions, ensure_ascii=False, separators=(",", ":")),
        "parse_ok": "true" if parse_ok else "false",
        "error_type": error_type,
        "raw_output_id": rid,
        "checkpoint_id": CHECKPOINT_ID,
        "prompt_version": PROMPT_VERSION,
        "runtime_sec": f"{runtime_sec:.6f}",
    }


def regions_json_is_list(value: Any) -> bool:
    try:
        parsed = json.loads(str(value or "[]"))
    except Exception:
        return False
    return isinstance(parsed, list)


def prune_unusable_cached_predictions(
    predictions: OrderedDict[str, dict[str, Any]],
    raw_rows: OrderedDict[str, dict[str, Any]],
) -> list[dict[str, str]]:
    dropped: list[dict[str, str]] = []
    for image_id, row in list(predictions.items()):
        rid = str(row.get("raw_output_id") or "")
        reason = ""
        if not regions_json_is_list(row.get("regions")):
            reason = "cached_regions_not_json_list"
        elif rid not in raw_rows and not SCORE_ONLY:
            reason = "cached_prediction_missing_raw_output"

        if reason:
            del predictions[image_id]
            dropped.append({"image_id": image_id, "raw_output_id": rid, "reason": reason})
    return dropped


def needs_model_inference(
    manifest_rows: list[dict[str, Any]],
    predictions: OrderedDict[str, dict[str, Any]],
    raw_rows: OrderedDict[str, dict[str, Any]],
) -> bool:
    if SCORE_ONLY:
        return False
    for record in manifest_rows:
        image_id = record["image_id"]
        if image_id in predictions:
            continue
        if raw_output_id(RUN_ID, image_id) not in raw_rows:
            return True
    return False


def prediction_counts(predictions: OrderedDict[str, dict[str, Any]]) -> dict[str, int]:
    return {
        "prediction_rows": len(predictions),
        "prediction_parse_ok": sum(1 for row in predictions.values() if str(row.get("parse_ok", "")).lower() == "true"),
        "prediction_parse_fail": sum(1 for row in predictions.values() if str(row.get("parse_ok", "")).lower() != "true"),
    }


def build_solution_rows(manifest_rows: list[dict[str, Any]]) -> list[dict[str, str]]:
    return [
        {
            "image": row["submission_image"],
            "regions": json.dumps(row.get("regions") or [], ensure_ascii=False, separators=(",", ":")),
        }
        for row in manifest_rows
    ]


def build_submission_rows(predictions: OrderedDict[str, dict[str, Any]], manifest_rows: list[dict[str, Any]]) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    for record in manifest_rows:
        pred = predictions.get(record["image_id"])
        rows.append({"image": record["submission_image"], "regions": str(pred.get("regions", "[]") if pred else "[]")})
    return rows


def write_runtime_summary(path: Path, runtime: dict[str, Any], prediction_count: int, parse_fail_count: int) -> dict[str, Any]:
    generated_count = int(runtime.get("generated_raw_count") or 0)
    summary = {
        "started_at": runtime["started_at"],
        "completed_at": runtime["completed_at"],
        "row_count_manifest": runtime["manifest_count"],
        "row_count_predictions": prediction_count,
        "parse_fail_count": parse_fail_count,
        "runtime_total_sec": runtime["runtime_total_sec"],
        "model_runtime_sec": runtime["model_runtime_sec"],
        "runtime_per_page_sec": runtime["runtime_total_sec"] / max(1, prediction_count),
        "model_runtime_per_generated_page_sec": runtime["model_runtime_sec"] / max(1, generated_count),
        "cached_prediction_count": int(runtime.get("cached_prediction_count") or 0),
        "generated_raw_count": generated_count,
        "reused_raw_count": int(runtime.get("reused_raw_count") or 0),
        "missing_image_count": int(runtime.get("missing_image_count") or 0),
        "inference_error_count": int(runtime.get("inference_error_count") or 0),
    }
    write_json(path, summary)
    return summary


def write_score(
    path: Path,
    manifest_rows: list[dict[str, Any]],
    predictions: OrderedDict[str, dict[str, Any]],
    raw_rows: OrderedDict[str, dict[str, Any]],
    runtime_summary: dict[str, Any],
) -> dict[str, Any]:
    invalid_region_rows = sum(1 for row in predictions.values() if not regions_json_is_list(row.get("regions")))
    breakdown = score_detailed(build_solution_rows(manifest_rows), build_submission_rows(predictions, manifest_rows))
    parse_fail_count = sum(1 for row in predictions.values() if str(row.get("parse_ok", "")).lower() != "true")
    missing_raw_count = sum(1 for row in predictions.values() if str(row.get("raw_output_id") or "") not in raw_rows)
    parse_fail_threshold = max(3, (len(manifest_rows) + 199) // 200)
    score = {
        "total_score": float(breakdown["composite_score"]),
        "detection_f1": float(breakdown["detection_f1"]),
        "class_acc": float(breakdown["class_acc"]),
        "region_cer": float(breakdown["region_cer"]),
        "page_cer": float(breakdown["page_cer"]),
        "row_count": len(predictions),
        "parse_fail_count": parse_fail_count,
        "parse_fail_threshold": parse_fail_threshold,
        "raw_output_count": len(raw_rows),
        "missing_raw_output_count": missing_raw_count,
        "invalid_region_json_count": invalid_region_rows,
        "runtime_total_sec": float(runtime_summary["runtime_total_sec"]),
        "runtime_per_page_sec": float(runtime_summary["runtime_per_page_sec"]),
        "metric_version": METRIC_VERSION,
        "metric_implementation": "standalone_internal_official_compatible",
        "source_manifest": ARTIFACT_VERSION,
        "source_manifest_path": display_path(MANIFEST_PATH),
        "pass_gate": {
            "prediction_row_count_matches_manifest": len(predictions) == len(manifest_rows),
            "regions_are_json_lists": invalid_region_rows == 0,
            "parse_fail_under_threshold": parse_fail_count <= parse_fail_threshold,
            "raw_outputs_complete": missing_raw_count == 0,
            "score_breakdown_complete": True,
        },
        "details": breakdown,
    }
    write_json(path, score)
    return score


def write_ablation_row(path: Path, score: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    row = {
        "stage": "B2_baseline",
        "run_id": RUN_ID,
        "checkpoint_id": CHECKPOINT_ID,
        "artifact_version": ARTIFACT_VERSION,
        "prompt_version": PROMPT_VERSION,
        "crop_mode": CROP_MODE,
        "routing_rule": "none",
        "decision": "keep",
        "decision_reason": "baseline anchor locked",
        "total_score": score.get("total_score"),
        "detection_f1": score.get("detection_f1"),
        "class_acc": score.get("class_acc"),
        "region_cer": score.get("region_cer"),
        "page_cer": score.get("page_cer"),
        "runtime_per_page_sec": score.get("runtime_per_page_sec"),
        "metric_version": score.get("metric_version"),
        "created_at": utc_now(),
    }

    if path.exists():
        frame = pd.read_csv(path, dtype=str, keep_default_na=False)
        if "run_id" in frame.columns:
            frame = frame[frame["run_id"] != RUN_ID]
        frame = pd.concat([pd.DataFrame([row]), frame], ignore_index=True)
    else:
        frame = pd.DataFrame([row])

    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def parse_existing_or_generate(
    manifest_rows: list[dict[str, Any]],
    prompt: PromptSettings,
    paths: dict[str, Path],
) -> tuple[OrderedDict[str, dict[str, Any]], OrderedDict[str, dict[str, Any]], dict[str, Any]]:
    predictions = read_existing_predictions(paths["predictions"])
    raw_rows = read_raw_outputs(paths["raw_outputs"])
    dropped_cached = prune_unusable_cached_predictions(predictions, raw_rows)

    runtime = {
        "started_at": utc_now(),
        "completed_at": None,
        "manifest_count": len(manifest_rows),
        "runtime_total_sec": 0.0,
        "model_runtime_sec": 0.0,
        "cached_prediction_count": len(predictions),
        "generated_raw_count": 0,
        "reused_raw_count": 0,
        "missing_image_count": 0,
        "inference_error_count": 0,
    }
    wall_start = time.time()

    counters = {
        "cached_predictions": len(predictions),
        "dropped_cached_predictions": len(dropped_cached),
        "raw_outputs_loaded": len(raw_rows),
        "generated": 0,
        "reused_raw": 0,
        "parse_ok_new": 0,
        "parse_fail_new": 0,
        "missing_image": 0,
        "inference_error": 0,
    }
    show_table("Cache Resume State", [{"counter": key, "value": value} for key, value in counters.items()])
    if dropped_cached:
        show_table("Dropped Cached Predictions", dropped_cached[:20])

    model = processor = None
    if needs_model_inference(manifest_rows, predictions, raw_rows):
        assert LORA_DIR is not None
        show_phase(
            "Loading Model",
            f"Base model: `{MODEL_ID}`\n\nLoRA: `{display_path(LORA_DIR)}`\n\nDevice: `{DEVICE}`, 4bit: `{LOAD_IN_4BIT}`",
        )
        load_start = time.time()
        model, processor = load_model_and_processor(LORA_DIR)
        show_phase("Model Loaded", f"Load time: `{format_duration(time.time() - load_start)}`. Starting page inference.")
    elif SCORE_ONLY:
        show_phase("Score Only Mode", "Model loading is skipped. The notebook will parse and score existing raw outputs/predictions.")
    else:
        show_phase("Model Load Skipped", "All missing predictions can be reused from cache or existing raw outputs.")

    progress = tqdm(manifest_rows, total=len(manifest_rows), desc="Stage B pages", unit="page")
    for index, record in enumerate(progress, start=1):
        image_id = record["image_id"]
        if image_id in predictions:
            progress.set_postfix(
                done=len(predictions),
                gen=counters["generated"],
                reused=counters["reused_raw"],
                fail=counters["parse_fail_new"],
                refresh=False,
            )
            continue

        rid = raw_output_id(RUN_ID, image_id)
        raw_row = raw_rows.get(rid)
        runtime_sec = 0.0
        should_log = (index == 1) or (index == len(manifest_rows)) or (LOG_EVERY > 0 and index % LOG_EVERY == 0)

        if raw_row is None:
            if SCORE_ONLY:
                if should_log:
                    tqdm.write(f"[image {index}/{len(manifest_rows)}] missing raw output in SCORE_ONLY mode: {image_id}")
                continue
            if model is None or processor is None:
                raise RuntimeError("Model was not loaded but a page still needs raw inference output.")

            image_path = resolve_image_path(record)
            raw_text = ""
            error_type = ""
            if LOG_IMAGE_START and should_log:
                tqdm.write(
                    f"[image {index}/{len(manifest_rows)}] start | "
                    f"file={record['submission_image']} | path={display_path(image_path)} | "
                    f"done={len(predictions)}"
                )
            if not image_path.exists():
                error_type = "missing_image_file"
                counters["missing_image"] += 1
                tqdm.write(f"[image {index}/{len(manifest_rows)}] missing image file: {display_path(image_path)}")
            else:
                start = time.time()
                try:
                    raw_text = generate_raw_text(
                        model,
                        processor,
                        image_path,
                        prompt,
                        log_label=f"[image {index}/{len(manifest_rows)}] model.generate",
                    )
                except Exception as exc:
                    error_type = "inference_error:" + str(exc).replace("\n", " ")[:240]
                    counters["inference_error"] += 1
                    tqdm.write(f"[image {index}/{len(manifest_rows)}] inference error: {error_type}")
                runtime_sec = time.time() - start
                runtime["model_runtime_sec"] += runtime_sec
                if LOG_IMAGE_DONE and should_log:
                    tqdm.write(
                        f"[image {index}/{len(manifest_rows)}] generated | "
                        f"runtime={runtime_sec:.2f}s | raw_chars={len(raw_text)}"
                    )

            raw_row = {
                "raw_output_id": rid,
                "image_id": image_id,
                "file_name": record["file_name"],
                "prompt_version": PROMPT_VERSION,
                "generation_params": generation_params(prompt),
                "raw_text": raw_text,
                "created_at": utc_now(),
                "runtime_sec": runtime_sec,
            }
            if error_type:
                raw_row["error_type"] = error_type
            append_jsonl(paths["raw_outputs"], raw_row)
            raw_rows[rid] = raw_row
            counters["generated"] += 1
        else:
            runtime_sec = float(raw_row.get("runtime_sec") or 0.0)
            counters["reused_raw"] += 1
            if should_log:
                tqdm.write(f"[image {index}/{len(manifest_rows)}] reused raw output | image_id={image_id}")

        raw_error = str(raw_row.get("error_type") or "")
        if raw_error == "missing_image_file":
            parsed = ParseResult([], False, "missing_image_file", [])
        elif raw_error.startswith("inference_error:"):
            parsed = ParseResult([], False, raw_error, [])
        else:
            parsed = parse_and_clean_regions(str(raw_row.get("raw_text") or ""), record)

        if parsed.parse_ok:
            counters["parse_ok_new"] += 1
        else:
            counters["parse_fail_new"] += 1

        if LOG_IMAGE_DONE and (should_log or not parsed.parse_ok):
            tqdm.write(
                f"[image {index}/{len(manifest_rows)}] parsed | "
                f"parse_ok={parsed.parse_ok} | regions={len(parsed.regions)} | "
                f"error={parsed.error_type or '-'} | total_done={len(predictions) + 1}"
            )

        predictions[image_id] = prediction_row(
            record=record,
            regions=parsed.regions,
            parse_ok=parsed.parse_ok,
            error_type=parsed.error_type,
            rid=rid,
            runtime_sec=runtime_sec,
        )

        progress.set_postfix(
            done=len(predictions),
            gen=counters["generated"],
            reused=counters["reused_raw"],
            ok=counters["parse_ok_new"],
            fail=counters["parse_fail_new"],
            refresh=False,
        )

        if len(predictions) % SAVE_EVERY == 0 or index == len(manifest_rows):
            write_predictions_csv(paths["predictions"], predictions, manifest_rows)
            tqdm.write(
                f"checkpoint {len(predictions)}/{len(manifest_rows)} | "
                f"image={record['submission_image']} | parse_ok={parsed.parse_ok} | "
                f"regions={len(parsed.regions)} | last_runtime={runtime_sec:.2f}s"
            )

    write_predictions_csv(paths["predictions"], predictions, manifest_rows)
    runtime["completed_at"] = utc_now()
    runtime["runtime_total_sec"] = time.time() - wall_start
    runtime["generated_raw_count"] = counters["generated"]
    runtime["reused_raw_count"] = counters["reused_raw"]
    runtime["missing_image_count"] = counters["missing_image"]
    runtime["inference_error_count"] = counters["inference_error"]

    totals = prediction_counts(predictions)
    show_table(
        "Inference Summary",
        [{"counter": key, "value": value} for key, value in counters.items()]
        + [{"counter": key, "value": value} for key, value in totals.items()]
        + [
            {"counter": "raw_outputs_total", "value": len(raw_rows)},
            {"counter": "runtime_total", "value": format_duration(runtime["runtime_total_sec"])},
            {"counter": "model_runtime", "value": format_duration(runtime["model_runtime_sec"])},
        ],
    )
    return predictions, raw_rows, runtime


## Main

Orchestrates the Stage B run end to end and updates final config status.


In [ ]:
# =============================================================================
# Main.
# =============================================================================


def main() -> int:
    paths = output_paths()
    paths["run_dir"].mkdir(parents=True, exist_ok=True)

    require_runtime_paths()
    manifest_rows = read_manifest(MANIFEST_PATH)
    prompt = load_prompt_settings()
    config = build_config(manifest_rows, prompt, paths)
    validate_stage_b_contract(config)
    write_or_validate_config(paths["config"], config)

    show_run_overview(config, prompt, paths, manifest_rows)

    predictions, raw_rows, runtime = parse_existing_or_generate(manifest_rows, prompt, paths)

    parse_fail_count = sum(1 for row in predictions.values() if str(row.get("parse_ok", "")).lower() != "true")
    runtime_summary = write_runtime_summary(paths["runtime_summary"], runtime, len(predictions), parse_fail_count)
    write_failed_rows(paths["failed_rows"], predictions, raw_rows)

    if len(predictions) != len(manifest_rows):
        print(f"Stage B incomplete: predictions={len(predictions)} manifest={len(manifest_rows)}", flush=True)
        return 2

    score = write_score(paths["score"], manifest_rows, predictions, raw_rows, runtime_summary)
    if WRITE_ABLATION_ROW:
        write_ablation_row(ABLATION_RESULTS_PATH, score)

    final_config = json.loads(paths["config"].read_text(encoding="utf-8"))
    final_config.update(
        {
            "status": "complete",
            "completed_at": utc_now(),
            "score_summary": {k: score[k] for k in ("total_score", "detection_f1", "class_acc", "region_cer", "page_cer")},
        }
    )
    write_json(paths["config"], final_config)

    show_score_summary(score, runtime_summary)
    show_artifact_outputs(paths)
    show_phase("Stage B Complete", "All required Stage B artifacts have been written for this run.")
    return 0


## Run Stage B

Keep `RUN_STAGE_B = False` while reviewing config. Set it to `True` only after `DATASET_ROOT`, `MODEL_ID`, `LORA_DIR`, and optionally `PROMPT_CONFIG_PATH` are correct for the Kaggle session. The run cell displays config tables, a `tqdm` page progress bar, checkpoint messages, score summary, pass-gate status, and artifact paths.


In [ ]:
RUN_STAGE_B = False

if RUN_STAGE_B:
    exit_code = main()
    print(f"main() returned {exit_code}")
else:
    print("Stage B is ready. Review CONFIG, set RUN_STAGE_B=True, then rerun this cell.")
